# 創薬仮説 バッチ生成ノートブック
**Batch Drug Discovery Hypothesis Generator**

1つの疾患に対して複数の遺伝子を順番に処理し、レポートを自動生成します。

- **Step 1**: LLM・言語設定
- **Step 2**: 疾患を選択
- **Step 3**: 対象遺伝子リストを入力
- **Step 4**: バッチ実行（全遺伝子を自動処理）

## Step 1: LLM・言語設定

In [ ]:
import sys
sys.path.insert(0, '.')

for mod in list(sys.modules.keys()):
    if 'llm' in mod:
        del sys.modules[mod]

from llm.ollama_client import OllamaClient

MODEL = 'llama3.1'   # 例: 'llama3.1:70b', 'gemma3:4b', 'phi4-mini'
LANG  = 'ja'        # 'ja' = 日本語 / 'en' = English

llm = OllamaClient(model=MODEL)
if llm.is_available():
    print(f'✓ Ollama ({MODEL}) 起動確認')
    print(f'  言語: {LANG}')
else:
    raise RuntimeError('⚠ Ollama に接続できません。ターミナルで ollama serve を実行してください。')

# ── コンテキスト情報量の設定 ──────────────────────────────
# 数値を変更するとLLMへ渡す情報量が変わります。
# バッチ処理では軽量設定が速度と品質のバランスに適しています。

CONTEXT_CONFIG = dict(
    max_papers       = 4,    # 論文数（PubMed）
    abstract_chars   = 300,  # アブストラクト 1件あたりの文字数
    max_drugs        = 6,    # 薬剤数（ChEMBL + OpenTargets）
    max_gwas         = 4,    # GWAS ヒット数
    max_clinvar      = 4,    # ClinVar バリアント数
    max_interactions = 10,   # PPI インタラクター数
    max_trials       = 4,    # 臨床試験数
    max_reactome     = 6,    # Reactome パスウェイ数
    gtex_top_n       = 3,    # GTEx 上位発現組織数
    hpa_top_n        = 5,    # Human Protein Atlas 組織数
    max_dgidb        = 5,    # DGIdb 薬剤-遺伝子相互作用数
    uniprot_chars    = 250,  # UniProt function 文字数
)

# 軽量モード（速度優先）
# CONTEXT_CONFIG = dict(max_papers=2, abstract_chars=150, max_drugs=3,
#                       max_gwas=2, max_clinvar=2, gtex_top_n=2, hpa_top_n=3,
#                       max_dgidb=3, max_trials=2, max_reactome=3,
#                       max_interactions=6, uniprot_chars=150)

from aggregator import DEFAULT_CONTEXT_CONFIG
print('\nコンテキスト設定:')
for k, v in CONTEXT_CONFIG.items():
    diff = f'  ← デフォルト: {DEFAULT_CONTEXT_CONFIG[k]}' if v != DEFAULT_CONTEXT_CONFIG[k] else ''
    print(f'  {k:<20} = {v}{diff}')

## Step 2: 疾患を選択

疾患名を入力して検索し、リストから選択してください。

In [ ]:
import requests
import ipywidgets as widgets
from IPython.display import display

OT_API = 'https://api.platform.opentargets.org/api/v4/graphql'

def _ot_search(keyword, entity):
    q = '''
    query ($q: String!, $e: [String!]) {
      search(queryString: $q, entityNames: $e, page: {index: 0, size: 15}) {
        hits { id name description entity }
      }
    }
    '''
    r = requests.post(OT_API, json={'query': q, 'variables': {'q': keyword, 'e': [entity]}}, timeout=15)
    r.raise_for_status()
    return [h for h in r.json()['data']['search']['hits'] if h['entity'] == entity]

selected_disease = {'id': None, 'name': None}

box  = widgets.Text(placeholder='疾患名を入力... (例: Duchenne muscular dystrophy)',
                    layout=widgets.Layout(width='440px'))
btn  = widgets.Button(description='検索', button_style='primary',
                      layout=widgets.Layout(width='70px'))
lst  = widgets.Select(options=[], rows=8,
                      layout=widgets.Layout(width='700px'))
stat = widgets.Label(value='疾患名を入力して「検索」を押してください')

def on_search(_):
    kw = box.value.strip()
    if not kw:
        stat.value = '⚠ キーワードを入力してください'; return
    stat.value = '検索中...'
    try:
        hits = _ot_search(kw, 'disease')
        if not hits:
            stat.value = f'「{kw}」に一致する疾患が見つかりませんでした'
            lst.options = []; return
        lst.options = [
            (f"{h['name']}  [{h['id']}]  {(h.get('description') or '')[:60]}", h)
            for h in hits
        ]
        stat.value = f'{len(hits)} 件見つかりました。リストから選択してください'
    except Exception as e:
        stat.value = f'エラー: {e}'

def on_select(change):
    val = change['new']
    if val:
        selected_disease['id']   = val['id']
        selected_disease['name'] = val['name']
        stat.value = f'✓ 選択済み: {val["name"]}  ({val["id"]})'

btn.on_click(on_search)
lst.observe(on_select, names='value')
display(widgets.VBox([widgets.HBox([box, btn]), lst, stat]))

## Step 3: 遺伝子リストを入力

対象遺伝子を入力してください。
- **1行1遺伝子** または **カンマ区切り**
- 入力後に「リストを確認」ボタンで内容を確認できます

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

gene_textarea = widgets.Textarea(
    placeholder='遺伝子名を入力（1行1遺伝子 または カンマ区切り）\n例:\nBRAF\nTP53\nEGFR',
    layout=widgets.Layout(width='400px', height='160px'),
)
confirm_btn = widgets.Button(description='リストを確認', button_style='info')
gene_out    = widgets.Output()

GENE_LIST = []

def parse_genes(text):
    import re
    genes = re.split(r'[,\n\r]+', text)
    return [g.strip().upper() for g in genes if g.strip()]

def on_confirm(_):
    global GENE_LIST
    GENE_LIST = parse_genes(gene_textarea.value)
    with gene_out:
        gene_out.clear_output()
        if GENE_LIST:
            rows = ''.join(f'<tr><td style="padding:4px 12px">{i+1}</td><td style="padding:4px 12px;font-weight:bold">{g}</td></tr>'
                           for i, g in enumerate(GENE_LIST))
            display(HTML(f'''
            <b>✓ {len(GENE_LIST)} 遺伝子を登録しました</b>
            <table style="margin-top:8px;border-collapse:collapse;border:1px solid #ddd">
              <thead><tr style="background:#f5f5f5">
                <th style="padding:4px 12px">#</th>
                <th style="padding:4px 12px">遺伝子</th>
              </tr></thead>
              <tbody>{rows}</tbody>
            </table>
            '''))
        else:
            display(HTML('<span style="color:red">⚠ 遺伝子が入力されていません</span>'))

confirm_btn.on_click(on_confirm)
display(widgets.VBox([gene_textarea, confirm_btn, gene_out]))

## Step 4: バッチ実行

全遺伝子を順番に処理します。各遺伝子のレポートは `reports/{GENE}_{DISEASE}/` に保存されます。

> ⚠ **注意**: LLM生成は遺伝子ごとに数分かかります。遺伝子数 × 生成時間を見込んでください。

In [ ]:
import sys, json
from datetime import datetime
from pathlib import Path
from IPython.display import display, Markdown, HTML

for mod in list(sys.modules.keys()):
    if any(x in mod for x in ('collectors', 'aggregator', 'hypothesis', 'network')):
        del sys.modules[mod]

from aggregator import collect_all, build_llm_context
from hypothesis import generate_hypothesis, generate_presentation_eval
import network as net_mod

# ── 入力チェック ──────────────────────────────────────────
if not selected_disease.get('name'):
    raise ValueError('⚠ Step 2 で疾患を選択してください')
if not GENE_LIST:
    raise ValueError('⚠ Step 3 で遺伝子を入力して「リストを確認」を押してください')

DISEASE    = selected_disease['name']
DISEASE_ID = selected_disease['id']

print(f'疾患: {DISEASE}  ({DISEASE_ID})')
print(f'対象遺伝子 ({len(GENE_LIST)}件): {", ".join(GENE_LIST)}')
print('=' * 60)

results_summary = []

for idx, GENE in enumerate(GENE_LIST, 1):
    print(f'\n[{idx}/{len(GENE_LIST)}] {GENE} × {DISEASE}')
    print('-' * 50)

    # ── データ収集 ────────────────────────────────────────
    print('  データ収集中...')
    try:
        raw_evidence = collect_all(
            GENE, DISEASE,
            verbose=False,
            disease_id=DISEASE_ID,
        )
        if raw_evidence.get('collection_errors'):
            print(f'  ⚠ 一部エラー: {list(raw_evidence["collection_errors"].keys())}')
    except Exception as e:
        print(f'  ✗ データ収集失敗: {e}')
        results_summary.append({'gene': GENE, 'status': f'データ収集失敗: {e}'})
        continue

    # ── PPIネットワーク ───────────────────────────────────
    try:
        ppi_graph = net_mod.build_ppi_network(GENE, use_biogrid=False)
        network_enrichment = net_mod.run_network_enrichment(ppi_graph) if ppi_graph else {}
    except Exception:
        ppi_graph = None
        network_enrichment = {}

    # ── コンテキスト生成 ──────────────────────────────────
    context = build_llm_context(raw_evidence, config=CONTEXT_CONFIG)
    if ppi_graph:
        context += '\n\n' + net_mod.network_summary_for_llm(
            ppi_graph, GENE, network_enrichment, max_partners=8, max_terms=10)
    print(f'  コンテキスト: {len(context):,} 文字')

    # ── 仮説生成 ─────────────────────────────────────────
    print('  仮説生成中... (ストリーミング)')
    try:
        buf = []
        def _cb(token):
            buf.append(token)
            print(token, end='', flush=True)

        hypothesis = generate_hypothesis(
            GENE, DISEASE, context, llm,
            lang=LANG,
            stream_callback=_cb,
        )
        print(f'\n  ✓ 仮説生成完了 ({len(hypothesis):,} 文字)')
    except Exception as e:
        print(f'\n  ✗ 仮説生成失敗: {e}')
        results_summary.append({'gene': GENE, 'status': f'仮説生成失敗: {e}'})
        continue

    # ── 評価カード ────────────────────────────────────────
    try:
        eval_result = generate_presentation_eval(GENE, DISEASE, context, llm, lang=LANG)
    except Exception:
        eval_result = {}

    # ── レポート保存 ──────────────────────────────────────
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    pair_dir  = Path('reports') / f'{GENE}_{DISEASE.replace(" ", "_")}'
    pair_dir.mkdir(parents=True, exist_ok=True)

    lang_suffix = 'JA' if LANG == 'ja' else 'EN'
    rpt_path = pair_dir / f'{timestamp}_{lang_suffix}.md'
    with open(rpt_path, 'w', encoding='utf-8') as f:
        f.write(f'# 創薬仮説レポート: {GENE} × {DISEASE}\n')
        f.write(f'生成日時: {datetime.now().isoformat()}  |  言語: {LANG}\n\n---\n\n')
        f.write(hypothesis)
        f.write('\n\n---\n\n## エビデンスコンテキスト\n\n')
        f.write(context)

    if eval_result:
        with open(pair_dir / f'{timestamp}_eval.json', 'w', encoding='utf-8') as f:
            json.dump(eval_result, f, ensure_ascii=False, indent=2)

    with open(pair_dir / f'{timestamp}_raw.json', 'w', encoding='utf-8') as f:
        json.dump(raw_evidence, f, ensure_ascii=False, indent=2, default=str)

    print(f'  ✓ 保存: {rpt_path}')
    results_summary.append({
        'gene': GENE,
        'status': '✓ 完了',
        'path': str(rpt_path),
        'overall': (eval_result.get('overall_confidence') or {}).get('rating', '-'),
    })

# ── 完了サマリー ──────────────────────────────────────────
print('\n' + '=' * 60)
print('バッチ完了')
rows = ''.join(
    f'<tr><td style="padding:5px 12px">{r["gene"]}</td>'
    f'<td style="padding:5px 12px">{r["status"]}</td>'
    f'<td style="padding:5px 12px">{r.get("overall","-")}</td>'
    f'<td style="padding:5px 12px;font-size:11px;color:#555">{r.get("path","-")}</td></tr>'
    for r in results_summary
)
display(HTML(f'''
<h3>バッチ結果サマリー — {DISEASE}</h3>
<table style="border-collapse:collapse;border:1px solid #ddd;width:100%">
  <thead><tr style="background:#f5f5f5">
    <th style="padding:5px 12px">遺伝子</th>
    <th style="padding:5px 12px">ステータス</th>
    <th style="padding:5px 12px">総合評価</th>
    <th style="padding:5px 12px">保存先</th>
  </tr></thead>
  <tbody>{rows}</tbody>
</table>
'''))